In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from datetime import datetime
import os, time

from scipy import stats
from sklearn.linear_model import LinearRegression
from typing import Optional
from fastf1.core import Session
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.stattools import grangercausalitytests
!pip install fastf1
import fastf1 as ff1
from fastf1 import plotting



os.chdir('/Users/jackson/Documents/ECON570')

In [ ]:
ff1.Cache.enable_cache("F1/f1_cache")  # or your existing path

years = range(2014, 2024)  # example window

records = []

for year in years:
    # You might want to be explicit about the race; here I assume Bahrain
    session = ff1.get_session(year, "Bahrain Grand Prix", "R")
    session.load()

    for drv in session.drivers:
        d = session.get_driver(drv)
        records.append({
            "year": year,
            "team_f1": d["TeamName"],     # e.g. 'Mercedes', 'McLaren'
            "driver_code": d["Abbreviation"],  # 'HAM', 'NOR', etc.
            "driver_name": d["FullName"]
        })

driver_year_df = pd.DataFrame(records)


core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.6.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
logger      WARNING 	Failed to load session info data!
core        WARNING 	Cannot load laps, telemetry, weather, and message data because the relevant API is not supported for this session.
core           INFO 	Finished loading data for 22 drivers: ['44', '6', '11', '3', '27', '1', '19', '77', '14', '7', '26', '8', '4', '13', '10', '17', '22', '20', '21', '9', '25', '99']
core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.6.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
logger      WARNING 	Failed to load session info data!
core        WARNING 	Cannot load laps, telemetry, weather, and message data because the relevant API is not supported for this session.
core           INFO 	Finished loa

In [10]:
team_lineups = (
    driver_year_df
    .groupby(["team_f1", "year"])["driver_code"]
    .apply(lambda x: tuple(sorted(set(x))))
    .reset_index(name="drivers")
)

In [11]:
team_lineups = team_lineups.sort_values(["team_f1", "year"])

team_lineups["drivers_prev"] = (
    team_lineups
    .groupby("team_f1")["drivers"]
    .shift(1)
)

team_lineups["driver_change"] = (
    (team_lineups["drivers"] != team_lineups["drivers_prev"])
    .astype(int)
)

In [17]:
name_map = {
    "Mercedes": "Mercedes",       # or 'Mercedes-Benz Grand Prix Ltd'
    "McLaren": "McLaren",
    "Ferrari": "Ferrari",
    "Red Bull Racing": "Red Bull",  # etc.
    # add others
}

team_lineups["team_clean_name"] = team_lineups["team_f1"].map(name_map)


In [20]:
profits_df = pd.read_csv("F1/f1_financial_data.csv")
profits_df.head()

profits_df = pd.read_csv("F1/f1_financial_data.csv")

profits_df = profits_df.rename(columns={
    "Team": "team_clean_name",
    "Year": "year",
    "Operating Profit (£m)": "profit"      # choose this as your main profit metric
})

profits_df.head()
panel = team_lineups.merge(
    profits_df,
    on=["team_clean_name", "year"],
    how="inner"
)

# Profit change vs previous season within a team
panel = panel.sort_values(["team_clean_name", "year"])

panel["profit_prev"] = (
    panel
    .groupby("team_clean_name")["profit"]
    .shift(1)
)

panel["profit_change"] = panel["profit"] - panel["profit_prev"]

panel.head()


,team_f1,year,drivers,drivers_prev,driver_change,team_clean_name,Legal Entity,Country,Revenue (£m),profit,Net Income (£m),Prize Money (£m),Employees,Source,profit_prev,profit_change
0,Ferrari,2019,"(LEC, VET)","(RAI, VET)",1,Ferrari,Gestione Sportiva (Ferrari S.p.A.),Italy,380,13,10,180,950,Ferrari Annual Report,NaN,NaN
1,Ferrari,2020,"(LEC, VET)","(LEC, VET)",0,Ferrari,Gestione Sportiva (Ferrari S.p.A.),Italy,350,9,7,165,930,NaN,13.0,-4.0
2,Ferrari,2021,"(LEC, SAI)","(LEC, VET)",1,Ferrari,Gestione Sportiva (Ferrari S.p.A.),Italy,365,12,9,170,940,NaN,9.0,3.0
3,Ferrari,2022,"(LEC, SAI)","(LEC, SAI)",0,Ferrari,Gestione Sportiva (Ferrari S.p.A.),Italy,392,15,12,178,960,NaN,12.0,3.0
4,Ferrari,2023,"(LEC, SAI)","(LEC, SAI)",0,Ferrari,Gestione Sportiva (Ferrari S.p.A.),Italy,405,17,13,180,970,NaN,15.0,2.0


In [29]:
# Example, adjust based on actual FastF1 team names you see in team_lineups['team_f1'].unique()

name_map = {
    # Sauber lineage
    "Sauber": "Alfa Romeo",
    "Alfa Romeo Racing": "Alfa Romeo",
    "Alfa Romeo": "Alfa Romeo",

    # Toro Rosso / AlphaTauri lineage
    "Toro Rosso": "AlphaTauri",
    "AlphaTauri": "AlphaTauri",

    # Renault / Lotus / Alpine lineage
    "Renault": "Alpine",
    "Lotus F1": "Alpine",
    "Alpine": "Alpine",

    # Force India / Racing Point / Aston Martin lineage
    "Force India": "Aston Martin",
    "Racing Point": "Aston Martin",
    "Aston Martin": "Aston Martin",

    # Red Bull lineage
    "Red Bull": "Red Bull Racing",
    "Red Bull Racing": "Red Bull Racing",

    # Marussia / Manor
    "Marussia": "Manor Marussia",
    "Manor Marussia": "Manor Marussia",

    # Stable names
    "Ferrari": "Ferrari",
    "McLaren": "McLaren",
    "Mercedes": "Mercedes",
    "Haas F1 Team": "Haas F1 Team",
    "Caterham": "Caterham",}

team_lineups["team_clean_name"] = (
    team_lineups["team_f1"].map(name_map).fillna(team_lineups["team_f1"])
)

team_lineups[["team_f1", "team_clean_name"]].drop_duplicates().sort_values("team_f1")



team_lineups["team_clean_name"] = team_lineups["team_f1"].map(name_map)


In [30]:
print(team_lineups.columns)
print(team_lineups[["team_f1", "team_clean_name"]].drop_duplicates().head(20))


Index(['team_f1', 'year', 'drivers', 'drivers_prev', 'driver_change',
       'team_clean_name'],
      dtype='object')
              team_f1  team_clean_name
0          Alfa Romeo       Alfa Romeo
2   Alfa Romeo Racing       Alfa Romeo
5          AlphaTauri       AlphaTauri
9              Alpine           Alpine
12       Aston Martin     Aston Martin
15           Caterham         Caterham
16            Ferrari          Ferrari
26        Force India     Aston Martin
31       Haas F1 Team     Haas F1 Team
39           Lotus F1           Alpine
41     Manor Marussia   Manor Marussia
43           Marussia   Manor Marussia
44            McLaren          McLaren
54           Mercedes         Mercedes
64       Racing Point     Aston Martin
66           Red Bull  Red Bull Racing
70    Red Bull Racing  Red Bull Racing
76            Renault           Alpine
81             Sauber       Alfa Romeo
86         Toro Rosso       AlphaTauri


In [31]:
    panel = team_lineups.merge(
    profits_df[["team_clean_name", "year", "profit", "Revenue (£m)", "Net Income (£m)"]],
    on=["team_clean_name", "year"],
    how="inner"
)


In [34]:
panel[["team_clean_name", "year", "drivers", "profit"]]


,team_clean_name,year,drivers,profit
0,AlphaTauri,2020,"(GAS, KVY)",1
1,AlphaTauri,2021,"(GAS, TSU)",2
2,AlphaTauri,2022,"(GAS, TSU)",3
3,AlphaTauri,2023,"(DEV, TSU)",4
4,Alpine,2021,"(ALO, OCO)",1
5,Alpine,2022,"(ALO, OCO)",3
6,Alpine,2023,"(GAS, OCO)",4
7,Aston Martin,2021,"(STR, VET)",2
8,Aston Martin,2022,"(HUL, STR)",5
9,Aston Martin,2023,"(ALO, STR)",8


In [36]:

# assuming `panel` is the df you printed
panel = panel.sort_values(["team_clean_name", "year"])

# previous drivers within each team
panel["drivers_prev"] = (
    panel
    .groupby("team_clean_name")["drivers"]
    .shift(1)
)

# 1 if lineup changed vs previous year, 0 otherwise
panel["driver_change"] = (panel["drivers"] != panel["drivers_prev"]).astype(int)

# previous year's profit and profit change
panel["profit_prev"] = (
    panel
    .groupby("team_clean_name")["profit"]
    .shift(1)
)

panel["profit_change"] = panel["profit"] - panel["profit_prev"]

# drop first year per team (no lag)
reg_data = panel.dropna(subset=["profit_prev"]).copy()


reg_data[["team_clean_name", "year", "drivers", "drivers_prev",
          "driver_change", "profit_prev", "profit", "profit_change"]]


model_fe = smf.ols(
    "profit_change ~ driver_change + C(team_clean_name) + C(year)",
    data=reg_data
).fit()

print(model_fe.summary())


                            OLS Regression Results                            
Dep. Variable:          profit_change   R-squared:                       0.598
Model:                            OLS   Adj. R-squared:                  0.361
Method:                 Least Squares   F-statistic:                     2.527
Date:                Thu, 20 Nov 2025   Prob (F-statistic):             0.0445
Time:                        11:30:54   Log-Likelihood:                -57.228
No. Observations:                  28   AIC:                             136.5
Df Residuals:                      17   BIC:                             151.1
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                                            coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------

R^2 is ok, but lets find what the driver's effect is. COntrol for points...

In [39]:

ff1.Cache.enable_cache("F1/f1_cache")  # or your cache path

years = sorted(panel["year"].unique())  # from your existing panel

records = []

for year in years:
    schedule = ff1.get_event_schedule(year)
    for rnd in schedule["RoundNumber"].unique():
        session = ff1.get_session(year, int(rnd), "R")
        session.load()
        res = session.results  # DataFrame

        for _, row in res.iterrows():
            records.append({
                "year": year,
                "team_f1": row["TeamName"],
                "points_race": row["Points"]
            })

team_points_raw = pd.DataFrame(records)

# sum across races to get season total
team_points = (
    team_points_raw
    .groupby(["team_f1", "year"], as_index=False)["points_race"]
    .sum()
    .rename(columns={"points_race": "points"})
)


core           INFO 	Loading data for Australian Grand Prix - Race [v3.6.1]
req            INFO 	Using cached data for session_info


req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core        WARNING 	Driver 77 completed the race distance 00:00.387000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['77', '44', '33', '5', '16', '20', '27', '7', '18', '26', '10', '4', '11', '23', '99', '63', '88', '8', '3', '55']
core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.6.1]
req    

ValueError: Cannot get testing event by round number!